# Python + RDBMS + SQL Training  
## Notebook 1 of 3: RDBMS Concepts, Schema Design, Keys and Basic SQL

### Duration
This notebook is designed for approximately **4 hours** of hands-on learning.

### What you will learn
- What an RDBMS is
- How tables, rows, columns and schema work
- How to design relationships between tables
- Primary key, foreign key, unique key and constraints
- Basic SQL commands: `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`
- How to run SQL using Python and SQLite in Google Colab


## 1. What is an RDBMS?

RDBMS stands for **Relational Database Management System**.

It stores data in tables. A table contains rows and columns.

Example:

| student_id | student_name | city |
|---|---|---|
| 1 | Rahul Kumar | Patna |
| 2 | Priya Singh | Kolkata |

In an RDBMS, tables can be connected with each other using keys.

Examples of RDBMS:
- SQLite
- MySQL
- PostgreSQL
- Oracle
- SQL Server
- SAP HANA Cloud


## 2. Why Do We Use Databases?

Without a database, data is often stored in Excel or files. This creates problems:

- Duplicate data
- Manual errors
- No strong relationship between records
- Difficult reporting
- Difficult access control
- Difficult multi-user update

A relational database solves these issues by storing data in a structured, connected and queryable format.


## 3. DBMS vs RDBMS

| Area | DBMS | RDBMS |
|---|---|---|
| Storage style | Files or records | Tables |
| Relationship support | Limited | Strong |
| Keys | May not be strict | Primary key and foreign key |
| Query language | Varies | SQL |
| Examples | File-based DBMS | SQLite, MySQL, PostgreSQL |

An RDBMS is the most common database model used in business applications.


## 4. Core Concepts

| Concept | Meaning |
|---|---|
| Table | Stores data in rows and columns |
| Row | One record |
| Column | One field |
| Schema | Structure of tables and relationships |
| Primary Key | Unique identifier for each row |
| Foreign Key | Connects one table to another table |
| Constraint | Rule applied to data |
| Relationship | Connection between entities |

Example: one student can enroll in many courses. This relationship is represented using tables and keys.


## 5. Case Study: Learning Academy Database

We will create a database for a training academy.

### Tables
1. `departments`
2. `students`
3. `instructors`
4. `courses`
5. `enrollments`
6. `payments`

### Relationships
- One department has many instructors.
- One department has many courses.
- One instructor teaches many courses.
- One student can enroll in many courses.
- One course can have many students.
- One enrollment can have one payment.


## 6. Relationship Diagram

```text
departments 1 ──── many instructors
departments 1 ──── many courses
instructors 1 ──── many courses
students 1 ──── many enrollments
courses 1 ──── many enrollments
enrollments 1 ──── 1 payments
```

The `enrollments` table is important because it connects students and courses.


## 7. SQL Data Types

| Type | Meaning | Example |
|---|---|---|
| INTEGER | Whole number | 101 |
| TEXT | Text value | Rahul |
| REAL | Decimal number | 4999.50 |
| DATE | Date value | 2026-02-01 |

SQLite is flexible with data types, but enterprise databases like PostgreSQL, Oracle and SAP HANA are stricter.


## 8. Keys and Constraints

### Primary Key
Uniquely identifies a row.

### Foreign Key
Connects one table with another.

### Unique Key
Prevents duplicate values.

### Composite Key
A key made from more than one column.

### Check Constraint
Restricts allowed values.

Example:
```sql
CHECK (status IN ('Active', 'Completed', 'Cancelled'))
```


## 9. Set Up SQLite in Python


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def connect(db_name):
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def exec_script(conn, script):
    conn.executescript(script)
    conn.commit()
    print("Script executed successfully.")

def execute(conn, sql, params=None):
    if params is None:
        params = []
    cur = conn.execute(sql, params)
    conn.commit()
    print(f"Rows affected: {cur.rowcount}")
    return cur

def query(conn, sql, params=None):
    if params is None:
        params = []
    return pd.read_sql_query(sql, conn, params=params)

def tables(conn):
    return query(conn, """
    SELECT name 
    FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """)

def show_schema(conn, table_name):
    print(f"\nSchema: {table_name}")
    display(query(conn, f"PRAGMA table_info({table_name});"))
    fk = query(conn, f"PRAGMA foreign_key_list({table_name});")
    if not fk.empty:
        print("Foreign Keys:")
        display(fk)

DB_NAME = "rdbms_notebook_01.db"
path = Path(DB_NAME)
if path.exists():
    path.unlink()

conn = connect(DB_NAME)
print("Connected to:", DB_NAME)


## 10. Create Tables

The following SQL script creates all tables with primary keys, foreign keys, unique constraints and check constraints.


In [ ]:
schema_sql = """
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);

CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT,
    registration_date DATE NOT NULL
);

CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY,
    instructor_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    department_id INTEGER NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    department_id INTEGER NOT NULL,
    instructor_id INTEGER NOT NULL,
    fee REAL NOT NULL CHECK (fee >= 0),
    level TEXT NOT NULL CHECK (level IN ('Beginner', 'Intermediate', 'Advanced')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrollment_date DATE NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Completed', 'Cancelled')),
    UNIQUE(student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    enrollment_id INTEGER NOT NULL UNIQUE,
    amount REAL NOT NULL CHECK (amount >= 0),
    payment_date DATE NOT NULL,
    payment_status TEXT NOT NULL CHECK (payment_status IN ('Paid', 'Pending', 'Refunded')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
"""
exec_script(conn, schema_sql)
tables(conn)


## 11. Inspect Table Schema

`PRAGMA table_info(table_name)` shows column details.

`PRAGMA foreign_key_list(table_name)` shows relationships.


In [ ]:
for table_name in tables(conn)["name"]:
    show_schema(conn, table_name)


## 12. Insert Sample Data

Parent tables must be inserted first.

Correct order:
1. departments
2. students
3. instructors
4. courses
5. enrollments
6. payments


In [ ]:
data_sql = """
INSERT INTO departments VALUES
(1, 'Data Science'),
(2, 'Software Engineering'),
(3, 'Business Analytics');

INSERT INTO students VALUES
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2026-01-05'),
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2026-01-06'),
(3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-07'),
(4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2026-01-10'),
(5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2026-01-12'),
(6, 'Neha Gupta', 'neha.gupta@example.com', 'Ranchi', '2026-01-13');

INSERT INTO instructors VALUES
(1, 'Dr. Meera Iyer', 'meera.iyer@example.com', 1),
(2, 'Arjun Sen', 'arjun.sen@example.com', 2),
(3, 'Kavita Rao', 'kavita.rao@example.com', 3);

INSERT INTO courses VALUES
(101, 'Python for Beginners', 2, 2, 4999, 'Beginner'),
(102, 'SQL and RDBMS Masterclass', 1, 1, 6999, 'Beginner'),
(103, 'Machine Learning Basics', 1, 1, 11999, 'Intermediate'),
(104, 'Business Dashboarding', 3, 3, 8999, 'Intermediate'),
(105, 'Advanced Data Engineering', 1, 1, 15999, 'Advanced');

INSERT INTO enrollments VALUES
(1001, 1, 101, '2026-02-01', 'Active'),
(1002, 1, 102, '2026-02-03', 'Completed'),
(1003, 2, 102, '2026-02-04', 'Active'),
(1004, 3, 103, '2026-02-05', 'Active'),
(1005, 4, 104, '2026-02-07', 'Cancelled'),
(1006, 5, 105, '2026-02-08', 'Active'),
(1007, 6, 101, '2026-02-08', 'Completed'),
(1008, 2, 104, '2026-02-09', 'Active');

INSERT INTO payments VALUES
(501, 1001, 4999, '2026-02-01', 'Paid'),
(502, 1002, 6999, '2026-02-03', 'Paid'),
(503, 1003, 6999, '2026-02-04', 'Pending'),
(504, 1004, 11999, '2026-02-05', 'Paid'),
(505, 1005, 0, '2026-02-07', 'Refunded'),
(506, 1006, 15999, '2026-02-08', 'Paid'),
(507, 1007, 4999, '2026-02-08', 'Paid'),
(508, 1008, 8999, '2026-02-09', 'Pending');
"""
exec_script(conn, data_sql)


## 13. Basic SELECT Query

`SELECT` is used to fetch data from a table.


In [ ]:
query(conn, "SELECT * FROM students;")


## 14. Select Specific Columns

In real projects, avoid `SELECT *` when you need only selected columns.


In [ ]:
query(conn, """
SELECT student_id, student_name, city
FROM students;
""")


## 15. WHERE Clause

`WHERE` filters rows.


In [ ]:
query(conn, """
SELECT student_id, student_name, city
FROM students
WHERE city = 'Patna';
""")


## 16. AND, OR and NOT

Use:
- `AND` when all conditions must be true
- `OR` when at least one condition must be true
- `NOT` to reverse a condition


In [ ]:
query(conn, """
SELECT course_id, course_name, fee, level
FROM courses
WHERE fee > 7000 AND level = 'Intermediate';
""")


In [ ]:
query(conn, """
SELECT student_name, city
FROM students
WHERE city = 'Patna' OR city = 'Kolkata';
""")


In [ ]:
query(conn, """
SELECT student_name, city
FROM students
WHERE NOT city = 'Patna';
""")


## 17. ORDER BY and LIMIT

`ORDER BY` sorts records.



`LIMIT` restricts output rows.


In [ ]:
query(conn, """
SELECT course_name, fee
FROM courses
ORDER BY fee DESC
LIMIT 3;
""")


## 18. DISTINCT

`DISTINCT` removes duplicates.


In [ ]:
query(conn, """
SELECT DISTINCT city
FROM students
ORDER BY city;
""")


## 19. INSERT Using Python Parameters

Parameterized queries use placeholders like `?`.

This is safer than joining strings manually.


In [ ]:
new_student = (7, "Rohan Das", "rohan.das@example.com", "Pune", "2026-02-15")

execute(conn, """
INSERT INTO students (student_id, student_name, email, city, registration_date)
VALUES (?, ?, ?, ?, ?);
""", new_student)

query(conn, "SELECT * FROM students WHERE student_id = 7;")


## 20. UPDATE Query


In [ ]:
execute(conn, """
UPDATE students
SET city = ?
WHERE student_id = ?;
""", ("Bengaluru", 7))

query(conn, "SELECT * FROM students WHERE student_id = 7;")


## 21. DELETE Query

Always use a `WHERE` condition while deleting specific rows.


In [ ]:
execute(conn, """
DELETE FROM students
WHERE student_id = ?;
""", (7,))

query(conn, "SELECT * FROM students ORDER BY student_id;")


## 22. Constraint Demo: Duplicate Email

The following insert fails because email is marked `UNIQUE`.


In [ ]:
try:
    execute(conn, """
    INSERT INTO students (student_id, student_name, email, city, registration_date)
    VALUES (?, ?, ?, ?, ?);
    """, (8, "Duplicate User", "rahul.kumar@example.com", "Patna", "2026-03-01"))
except Exception as e:
    print("Error:", e)


## 23. Constraint Demo: Invalid Foreign Key

The following insert fails because student ID `999` does not exist.


In [ ]:
try:
    execute(conn, """
    INSERT INTO enrollments (enrollment_id, student_id, course_id, enrollment_date, status)
    VALUES (?, ?, ?, ?, ?);
    """, (2001, 999, 101, "2026-03-01", "Active"))
except Exception as e:
    print("Error:", e)


## 24. Practice Exercise

Write SQL queries for the following:

1. Show all courses.
2. Show all beginner courses.
3. Show all students from Patna.
4. Show the highest fee course.
5. Show all pending payments.
6. Show students registered after `2026-01-08`.
7. Show all courses sorted by fee from low to high.


In [ ]:
# Exercise 1
query(conn, """
SELECT * FROM courses;
""")


In [ ]:
# Exercise 2
query(conn, """
SELECT * FROM courses
WHERE level = 'Beginner';
""")


In [ ]:
# Exercise 3
query(conn, """
SELECT * FROM payments
WHERE payment_status = 'Pending';
""")


## 25. Notebook 1 Summary

You learned:

- What RDBMS means
- How schema design works
- How primary keys and foreign keys connect tables
- How constraints protect data quality
- How to create tables
- How to insert, read, update and delete records
- How Python can execute SQL queries

Next notebook: joins, aggregations, subqueries, CTEs, window functions and transactions.
